# 02 - Data Profiling with DQX
**Part 1 of the assignment**: Profile the data and document findings.

Goal: Understand structure, quality, characteristics, and find common attributes for merging.

In [0]:
%run ./00_setup_config

# 00 - Setup & Configuration
This notebook sets up the database, defines paths, and installs required libraries for the Food Inspection project.

All file paths are parameterized using widgets — no hardcoded values.

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## Widget Parameters

Volume Path      : /Volumes/workspace/food_inspection/raw_data
Raw Chicago Path : /Volumes/workspace/food_inspection/raw_data/Food_Inspections_20260411.csv
Raw Dallas Path  : /Volumes/workspace/food_inspection/raw_data/Restaurant_and_Food_Establishment_Inspections_(October_2016_to_January_2024)_20260411.csv
Database Name    : food_inspection


## Create Database

Database 'food_inspection' is ready.


## Verify Raw Data Files Exist

Raw data files in Volume:
  Food_Inspections_20260411.csv (330.70 MB)
  Restaurant_and_Food_Establishment_Inspections_(October_2016_to_January_2024)_20260411.csv (192.79 MB)


Configuration ready. All path variables are available via %run.


In [0]:
spark.sql(f"USE {DATABASE_NAME}")

DataFrame[]

## 1. Load Bronze Tables

In [0]:
df_chicago = spark.table(f"{DATABASE_NAME}.bronze_chicago_inspections")
df_dallas = spark.table(f"{DATABASE_NAME}.bronze_dallas_inspections")

## 2. Chicago Data Profiling

### 2.1 Schema Overview

In [0]:
print(f"Chicago - Rows: {df_chicago.count()}, Columns: {len(df_chicago.columns)}")
print("\nColumns:")
for c in df_chicago.columns:
    print(f"  - {c}")

Chicago - Rows: 308431, Columns: 20

Columns:
  - Inspection_ID
  - DBA_Name
  - AKA_Name
  - License
  - Facility_Type
  - Risk
  - Address
  - City
  - State
  - Zip
  - Inspection_Date
  - Inspection_Type
  - Results
  - Violations
  - Latitude
  - Longitude
  - Location
  - data_source_name
  - ingest_timestamp
  - etl_job_id


In [0]:
df_chicago.printSchema()

root
 |-- Inspection_ID: integer (nullable = true)
 |-- DBA_Name: string (nullable = true)
 |-- AKA_Name: string (nullable = true)
 |-- License: integer (nullable = true)
 |-- Facility_Type: string (nullable = true)
 |-- Risk: string (nullable = true)
 |-- Address: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zip: integer (nullable = true)
 |-- Inspection_Date: date (nullable = true)
 |-- Inspection_Type: string (nullable = true)
 |-- Results: string (nullable = true)
 |-- Violations: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Location: string (nullable = true)
 |-- data_source_name: string (nullable = true)
 |-- ingest_timestamp: timestamp (nullable = true)
 |-- etl_job_id: string (nullable = true)



### 2.2 Null Analysis - Chicago

In [0]:
from pyspark.sql.functions import col, count, when, isnan, round as spark_round, countDistinct, min, max, avg
from pyspark.sql.types import StringType

def null_or_empty(c, dtype):
    """Check for null, and also empty string if column is StringType."""
    if isinstance(dtype, StringType):
        return col(c).isNull() | (col(c) == "")
    return col(c).isNull()

# Null counts and percentages for Chicago
chicago_total = df_chicago.count()

null_analysis_chicago = df_chicago.select([
    count(when(null_or_empty(c, df_chicago.schema[c].dataType), c)).alias(c)
    for c in df_chicago.columns
])

print("=== CHICAGO: Null/Empty Counts ===")
display(null_analysis_chicago)

=== CHICAGO: Null/Empty Counts ===


Inspection_ID,DBA_Name,AKA_Name,License,Facility_Type,Risk,Address,City,State,Zip,Inspection_Date,Inspection_Type,Results,Violations,Latitude,Longitude,Location,data_source_name,ingest_timestamp,etl_job_id
0,0,2420,19,5323,88,0,189,67,42,0,1,0,86503,1051,1051,1051,0,0,0


In [0]:
# Null percentage
null_pct_chicago = df_chicago.select([
    spark_round((count(when(null_or_empty(c, df_chicago.schema[c].dataType), c)) / chicago_total * 100), 2).alias(c)
    for c in df_chicago.columns
])

print("=== CHICAGO: Null/Empty Percentage ===")
display(null_pct_chicago)

=== CHICAGO: Null/Empty Percentage ===


Inspection_ID,DBA_Name,AKA_Name,License,Facility_Type,Risk,Address,City,State,Zip,Inspection_Date,Inspection_Type,Results,Violations,Latitude,Longitude,Location,data_source_name,ingest_timestamp,etl_job_id
0.0,0.0,0.78,0.01,1.73,0.03,0.0,0.06,0.02,0.01,0.0,0.0,0.0,28.05,0.34,0.34,0.34,0.0,0.0,0.0


### 2.3 Distinct Value Counts - Chicago

In [0]:
distinct_counts_chicago = df_chicago.select([
    countDistinct(col(c)).alias(c) for c in df_chicago.columns
])

print("=== CHICAGO: Distinct Value Counts ===")
display(distinct_counts_chicago)

=== CHICAGO: Distinct Value Counts ===


Inspection_ID,DBA_Name,AKA_Name,License,Facility_Type,Risk,Address,City,State,Zip,Inspection_Date,Inspection_Type,Results,Violations,Latitude,Longitude,Location,data_source_name,ingest_timestamp,etl_job_id
308431,34626,32950,48312,520,4,33062,89,6,132,4095,110,7,220306,18813,18813,18813,1,1,1


### 2.4 Key Column Distributions - Chicago

In [0]:
# Inspection Results distribution
print("=== Chicago: Inspection Results Distribution ===")
display(
    df_chicago.groupBy("Results")
    .count()
    .orderBy(col("count").desc())
)

=== Chicago: Inspection Results Distribution ===


Results,count
Pass,159444
Fail,59470
Pass w/ Conditions,45944
Out of Business,25470
No Entry,13675
Not Ready,4334
Business Not Located,94


In [0]:
# Inspection Type distribution
print("=== Chicago: Inspection Type Distribution ===")
display(
    df_chicago.groupBy("Inspection_Type")
    .count()
    .orderBy(col("count").desc())
)

=== Chicago: Inspection Type Distribution ===


Inspection_Type,count
Canvass,160034
License,41130
Canvass Re-Inspection,34376
Complaint,28951
License Re-Inspection,12597
Complaint Re-Inspection,12069
Short Form Complaint,9074
Non-Inspection,5331
Suspected Food Poisoning,1000
Consultation,679


In [0]:
# Risk distribution
print("=== Chicago: Risk Category Distribution ===")
display(
    df_chicago.groupBy("Risk")
    .count()
    .orderBy(col("count").desc())
)

=== Chicago: Risk Category Distribution ===


Risk,count
Risk 1 (High),228865
Risk 2 (Medium),55343
Risk 3 (Low),24048
null,88
All,87


In [0]:
# Facility Type distribution (top 20)
print("=== Chicago: Top 20 Facility Types ===")
display(
    df_chicago.groupBy("Facility_Type")
    .count()
    .orderBy(col("count").desc())
    .limit(20)
)

=== Chicago: Top 20 Facility Types ===


Facility_Type,count
Restaurant,209122
Grocery Store,37140
School,19260
Children's Services Facility,7435
null,5323
Bakery,4461
Daycare Above and Under 2 Years,4081
Daycare (2 - 6 Years),3247
Long Term Care,2377
Catering,1902


In [0]:
# Zip code distribution (top 20)
print("=== Chicago: Top 20 Zip Codes ===")
display(
    df_chicago.groupBy("Zip")
    .count()
    .orderBy(col("count").desc())
    .limit(20)
)

=== Chicago: Top 20 Zip Codes ===


Zip,count
60614,11467
60647,11356
60657,10481
60611,9737
60618,9693
60622,9504
60608,9416
60625,8763
60607,8572
60639,8262


### 2.5 Chicago Violations Analysis
Note: Chicago violations are stored as a single pipe-delimited string.

In [0]:
from pyspark.sql.functions import split, explode, trim, length, regexp_extract

# Sample violations to understand structure
display(
    df_chicago.select("Inspection_ID", "Violations")
    .filter(col("Violations").isNotNull())
    .limit(5)
)

Inspection_ID,Violations
2633665,"10. ADEQUATE HANDWASHING SINKS PROPERLY SUPPLIED AND ACCESSIBLE - Comments: OBSERVED HAND WASHING ONLY SIGNAGE NEED FOR THE ONE COMPARTMENT SINK THAT THEY WILL USE AS A HAND WASHING SINK FOR THE REAR PREP AREA JUST TO THE RIGHT OF THE WALK IN COOLING UNIT. INSTRUCTED TO OBTAIN AND PLACE AT THE SINK UNIT. | 38. INSECTS, RODENTS, & ANIMALS NOT PRESENT - Comments: OBSERVED A GAP AT THE BOTTOM LEFT OF THE SINGLE EXIT DOOR OF ABOUT 1/4 INCH GAP. INSTRUCTED TO REPLACE OR ADJUST AS A MEANS OF PREVENTING POSSIBLE PEST AND OR INSECT ACTIVITY. | 51. PLUMBING INSTALLED; PROPER BACKFLOW DEVICES - Comments: OBSERVED A LEAK AT THE RIGHT FAUCET OF THE FOUR COMPARTMENT SINK LOCATED AT THE BAR. INSTRUCTED TO REPAIR AND MAINTAIN IN GOOD WORKING ORDER."
2633654,"36. THERMOMETERS PROVIDED & ACCURATE - Comments: OBSERVED BROKEN THERMOMETERS INSIDE UPRIGHT REFRIGERATION UNITS IN FRONT PREP AREA. INSTRUCTED TO REPLACE AND MAINTAIN. | 47. FOOD & NON-FOOD CONTACT SURFACES CLEANABLE, PROPERLY DESIGNED, CONSTRUCTED & USED - Comments: OBSERVED RUSTED SHELVING RACKS INSIDE UPRIGHT COCA COLA COOLING EQUIPMENT. INSTRUCTED TO RESURFACE OR REPLACE TO MAINTAIN IN GOOD REPAIR. | 53. TOILET FACILITIES: PROPERLY CONSTRUCTED, SUPPLIED, & CLEANED - Comments: OBSERVED GARBAGE RECEPTACLE IN WOMENS TOILETROOM WITHOUT COVERING. INSTRUCTED TO PROVIDE AND MAINTAIN. | 55. PHYSICAL FACILITIES INSTALLED, MAINTAINED & CLEAN - Comments: OBSERVED FLOORS IN REAR PREP AREA AROUND ALL HEAVY COOKING EQUIPMENT AND UNDER 3 COMPARTMENT SINK IN NEED OF CLEANING. INSTRUCTED TO PROVIDE AND MAINTAIN FLOOR. | 56. ADEQUATE VENTILATION & LIGHTING; DESIGNATED AREAS USED - Comments: OBSERVED INADEQUATE LIGHTING IN REAR PREP AREA. INSTRUCTED TO REPLACE BURNED OUT BULBS AND MAINTAIN"
2633653,"47. FOOD & NON-FOOD CONTACT SURFACES CLEANABLE, PROPERLY DESIGNED, CONSTRUCTED & USED - Comments: 4-501.11 OBSERVED DAMAGED GASKET ON REACH-IN COOLER DOOR NEAR SERVING AREA. INSTRUCTED MANAGER TO REPLACE COOLER DOOR GASKET AND MAINTAIN."
2633638,"38. INSECTS, RODENTS, & ANIMALS NOT PRESENT - Comments: OBSERVED OVER 20 RAT DROPPINGS SCATTERED ON FLOOR UNDER DISPLAY SHELVES, NEAR WATER HEATER AND BETWEEN SHELVING IN SALES, PREP AND STORAGE AREAS. INSTRUCTED MANAGER TO CALL AN EXTERMINATOR FOR SERVICE CLEAN AND SANITIZE ALL AREAS. PRIORITY FOUNDATION 7-38-020(A) CITATION ISSUED. | 51. PLUMBING INSTALLED; PROPER BACKFLOW DEVICES - Comments: 5-205.15 OBSERVED LEAK UNDER UTILITY SINK AND HANDWASHING SINK INSIDE WASHROOM. INSTRUCTED MANAGER TO REPAIR AND MAINTAIN. | 55. PHYSICAL FACILITIES INSTALLED, MAINTAINED & CLEAN - Comments: 6-501.114 OBSERVED UNUSED EQUIPMENT AND CRATES STORED OUTSIDE AND IN REAR STORAGE AREAS. INSTRUCTED MANAGER TO REMOVE ALL UNNECESSARY ITEMS TO PREVENT PEST HARBORAGE."
2633566,56. ADEQUATE VENTILATION & LIGHTING; DESIGNATED AREAS USED - Comments: OBSERVED EXCESSIVE DUST BUILD-UP ON THE FILTERS AT KITCHEN EXHAUST HOOD SYSTEM.INSTRUCTED TO CLEAN IN DETAIL AND MAINTAIN.


In [0]:
# Count inspections with no violations
no_violations_chicago = df_chicago.filter(col("Violations").isNull() | (col("Violations") == "")).count()
print(f"Chicago inspections with NO violations: {no_violations_chicago} ({no_violations_chicago/chicago_total*100:.2f}%)")
print(f"Chicago inspections WITH violations: {chicago_total - no_violations_chicago}")

Chicago inspections with NO violations: 86503 (28.05%)
Chicago inspections WITH violations: 221928


## 3. Dallas Data Profiling

### 3.1 Schema Overview

In [0]:
print(f"Dallas - Rows: {df_dallas.count()}, Columns: {len(df_dallas.columns)}")
print("\nColumns:")
for c in df_dallas.columns:
    print(f"  - {c}")

Dallas - Rows: 78984, Columns: 117

Columns:
  - Restaurant_Name
  - Inspection_Type
  - Inspection_Date
  - Inspection_Score
  - Street_Number
  - Street_Name
  - Street_Direction
  - Street_Type
  - Street_Unit
  - Street_Address
  - Zip_Code
  - Violation_Description_1
  - Violation_Points_1
  - Violation_Detail_1
  - Violation_Memo_1
  - Violation_Description_2
  - Violation_Points_2
  - Violation_Detail_2
  - Violation_Memo_2
  - Violation_Description_3
  - Violation_Points_3
  - Violation_Detail_3
  - Violation_Memo_3
  - Violation_Description_4
  - Violation_Points_4
  - Violation_Detail_4
  - Violation_Memo_4
  - Violation_Description_5
  - Violation_Points_5
  - Violation_Detail_5
  - Violation_Memo_5
  - Violation_Description_6
  - Violation_Points_6
  - Violation_Detail_6
  - Violation_Memo_6
  - Violation_Description_7
  - Violation_Points_7
  - Violation_Detail_7
  - Violation_Memo_7
  - Violation_Description_8
  - Violation_Points_8
  - Violation_Detail_8
  - Violation_Me

In [0]:
df_dallas.printSchema()

root
 |-- Restaurant_Name: string (nullable = true)
 |-- Inspection_Type: string (nullable = true)
 |-- Inspection_Date: date (nullable = true)
 |-- Inspection_Score: integer (nullable = true)
 |-- Street_Number: integer (nullable = true)
 |-- Street_Name: string (nullable = true)
 |-- Street_Direction: string (nullable = true)
 |-- Street_Type: string (nullable = true)
 |-- Street_Unit: string (nullable = true)
 |-- Street_Address: string (nullable = true)
 |-- Zip_Code: string (nullable = true)
 |-- Violation_Description_1: string (nullable = true)
 |-- Violation_Points_1: integer (nullable = true)
 |-- Violation_Detail_1: string (nullable = true)
 |-- Violation_Memo_1: string (nullable = true)
 |-- Violation_Description_2: string (nullable = true)
 |-- Violation_Points_2: integer (nullable = true)
 |-- Violation_Detail_2: string (nullable = true)
 |-- Violation_Memo_2: string (nullable = true)
 |-- Violation_Description_3: string (nullable = true)
 |-- Violation_Points_3: integer (n

### 3.2 Null Analysis - Dallas

In [0]:
dallas_total = df_dallas.count()

null_analysis_dallas = df_dallas.select([
    count(when(null_or_empty(c, df_dallas.schema[c].dataType), c)).alias(c)
    for c in df_dallas.columns
])

print("=== DALLAS: Null/Empty Counts (key columns) ===")
display(null_analysis_dallas)

=== DALLAS: Null/Empty Counts (key columns) ===


Restaurant_Name,Inspection_Type,Inspection_Date,Inspection_Score,Street_Number,Street_Name,Street_Direction,Street_Type,Street_Unit,Street_Address,Zip_Code,Violation_Description_1,Violation_Points_1,Violation_Detail_1,Violation_Memo_1,Violation_Description_2,Violation_Points_2,Violation_Detail_2,Violation_Memo_2,Violation_Description_3,Violation_Points_3,Violation_Detail_3,Violation_Memo_3,Violation_Description_4,Violation_Points_4,Violation_Detail_4,Violation_Memo_4,Violation_Description_5,Violation_Points_5,Violation_Detail_5,Violation_Memo_5,Violation_Description_6,Violation_Points_6,Violation_Detail_6,Violation_Memo_6,Violation_Description_7,Violation_Points_7,Violation_Detail_7,Violation_Memo_7,Violation_Description_8,Violation_Points_8,Violation_Detail_8,Violation_Memo_8,Violation_Description_9,Violation_Points_9,Violation_Detail_9,Violation_Memo_9,Violation_Description_10,Violation_Points_10,Violation_Detail_10,Violation_Memo_10,Violation_Description_11,Violation_Points_11,Violation_Detail_11,Violation_Memo_11,Violation_Description_12,Violation_Points_12,Violation_Detail_12,Violation_Memo_12,Violation_Description_13,Violation_Points_13,Violation_Detail_13,Violation_Memo_13,Violation_Description_14,Violation_Points_14,Violation_Detail_14,Violation_Memo_14,Violation_Description_15,Violation_Points_15,Violation_Detail_15,Violation_Memo_15,Violation_Description_16,Violation_Points_16,Violation_Detail_16,Violation_Memo_16,Violation_Description_17,Violation_Points_17,Violation_Detail_17,Violation_Memo_17,Violation_Description_18,Violation_Points_18,Violation_Detail_18,Violation_Memo_18,Violation_Description_19,Violation_Points_19,Violation_Detail_19,Violation_Memo_19,Violation_Description_20,Violation_Points_20,Violation_Detail_20,Violation_Memo_20,Violation_Description_21,Violation_Points_21,Violation_Detail_21,Violation_Memo_21,Violation_Description_22,Violation_Points_22,Violation_Detail_22,Violation_Memo_22,Violation_Description_23,Violation_Points_23,Violation_Detail_23,Violation_Memo_23,Violation_Description_24,Violation_Points_24,Violation_Detail_24,Violation_Memo_24,Violation_Description_25,Violation_Points_25,Violation_Detail_25,Violation_Memo_25,Inspection_Month,Inspection_Year,Lat_Long_Location,data_source_name,ingest_timestamp,etl_job_id
11,0,0,0,0,0,52898,1674,50804,0,0,6579,6579,7133,22841,14129,14129,14719,29221,23071,23071,23599,35994,31954,31954,32454,42579,40179,40179,40658,48710,47592,47592,47986,54291,54155,54155,54488,59171,59822,59822,60085,63539,64648,64648,64865,67248,68717,68717,68878,70383,72098,72098,72199,73207,74711,74711,74796,75466,76270,76270,76325,76738,77075,77075,77105,77405,77639,77639,77664,77885,78089,78089,78106,78239,78431,78431,78443,78534,78630,78630,78638,78692,78776,78776,78779,78815,78852,78852,78858,78872,78910,78910,78915,78921,78951,78951,78954,78953,78966,78966,78966,78967,78972,78972,78972,78972,78978,78978,78979,78978,0,0,8638,0,0,0


In [0]:
# Null percentage - Dallas
null_pct_dallas = df_dallas.select([
    spark_round((count(when(null_or_empty(c, df_dallas.schema[c].dataType), c)) / dallas_total * 100), 2).alias(c)
    for c in df_dallas.columns
])

print("=== DALLAS: Null/Empty Percentage ===")
display(null_pct_dallas)

=== DALLAS: Null/Empty Percentage ===


Restaurant_Name,Inspection_Type,Inspection_Date,Inspection_Score,Street_Number,Street_Name,Street_Direction,Street_Type,Street_Unit,Street_Address,Zip_Code,Violation_Description_1,Violation_Points_1,Violation_Detail_1,Violation_Memo_1,Violation_Description_2,Violation_Points_2,Violation_Detail_2,Violation_Memo_2,Violation_Description_3,Violation_Points_3,Violation_Detail_3,Violation_Memo_3,Violation_Description_4,Violation_Points_4,Violation_Detail_4,Violation_Memo_4,Violation_Description_5,Violation_Points_5,Violation_Detail_5,Violation_Memo_5,Violation_Description_6,Violation_Points_6,Violation_Detail_6,Violation_Memo_6,Violation_Description_7,Violation_Points_7,Violation_Detail_7,Violation_Memo_7,Violation_Description_8,Violation_Points_8,Violation_Detail_8,Violation_Memo_8,Violation_Description_9,Violation_Points_9,Violation_Detail_9,Violation_Memo_9,Violation_Description_10,Violation_Points_10,Violation_Detail_10,Violation_Memo_10,Violation_Description_11,Violation_Points_11,Violation_Detail_11,Violation_Memo_11,Violation_Description_12,Violation_Points_12,Violation_Detail_12,Violation_Memo_12,Violation_Description_13,Violation_Points_13,Violation_Detail_13,Violation_Memo_13,Violation_Description_14,Violation_Points_14,Violation_Detail_14,Violation_Memo_14,Violation_Description_15,Violation_Points_15,Violation_Detail_15,Violation_Memo_15,Violation_Description_16,Violation_Points_16,Violation_Detail_16,Violation_Memo_16,Violation_Description_17,Violation_Points_17,Violation_Detail_17,Violation_Memo_17,Violation_Description_18,Violation_Points_18,Violation_Detail_18,Violation_Memo_18,Violation_Description_19,Violation_Points_19,Violation_Detail_19,Violation_Memo_19,Violation_Description_20,Violation_Points_20,Violation_Detail_20,Violation_Memo_20,Violation_Description_21,Violation_Points_21,Violation_Detail_21,Violation_Memo_21,Violation_Description_22,Violation_Points_22,Violation_Detail_22,Violation_Memo_22,Violation_Description_23,Violation_Points_23,Violation_Detail_23,Violation_Memo_23,Violation_Description_24,Violation_Points_24,Violation_Detail_24,Violation_Memo_24,Violation_Description_25,Violation_Points_25,Violation_Detail_25,Violation_Memo_25,Inspection_Month,Inspection_Year,Lat_Long_Location,data_source_name,ingest_timestamp,etl_job_id
0.01,0.0,0.0,0.0,0.0,0.0,66.97,2.12,64.32,0.0,0.0,8.33,8.33,9.03,28.92,17.89,17.89,18.64,37.0,29.21,29.21,29.88,45.57,40.46,40.46,41.09,53.91,50.87,50.87,51.48,61.67,60.26,60.26,60.75,68.74,68.56,68.56,68.99,74.92,75.74,75.74,76.07,80.45,81.85,81.85,82.12,85.14,87.0,87.0,87.21,89.11,91.28,91.28,91.41,92.69,94.59,94.59,94.7,95.55,96.56,96.56,96.63,97.16,97.58,97.58,97.62,98.0,98.3,98.3,98.33,98.61,98.87,98.87,98.89,99.06,99.3,99.3,99.32,99.43,99.55,99.55,99.56,99.63,99.74,99.74,99.74,99.79,99.83,99.83,99.84,99.86,99.91,99.91,99.91,99.92,99.96,99.96,99.96,99.96,99.98,99.98,99.98,99.98,99.98,99.98,99.98,99.98,99.99,99.99,99.99,99.99,0.0,0.0,10.94,0.0,0.0,0.0


### 3.3 Key Column Distributions - Dallas

In [0]:
# Inspection Type distribution
print("=== Dallas: Inspection Type Distribution ===")
display(
    df_dallas.groupBy("Inspection_Type")
    .count()
    .orderBy(col("count").desc())
)

=== Dallas: Inspection Type Distribution ===


Inspection_Type,count
Routine,78019
Follow-up,935
Complaint,30


In [0]:
# Inspection Score statistics
print("=== Dallas: Inspection Score Statistics ===")
display(
    df_dallas.select(
        min("Inspection_Score").alias("min_score"),
        max("Inspection_Score").alias("max_score"),
        avg("Inspection_Score").alias("avg_score"),
        count(when(col("Inspection_Score") > 100, True)).alias("scores_above_100"),
        count(when(col("Inspection_Score").isNull(), True)).alias("null_scores")
    )
)

=== Dallas: Inspection Score Statistics ===


min_score,max_score,avg_score,scores_above_100,null_scores
-26,100,90.86361794793882,0,0


In [0]:
# Score distribution
print("=== Dallas: Score Distribution (binned) ===")
display(
    df_dallas.withColumn("score_bin",
        when(col("Inspection_Score") >= 90, "90-100 (Excellent)")
        .when(col("Inspection_Score") >= 80, "80-89 (Good)")
        .when(col("Inspection_Score") >= 70, "70-79 (Fair)")
        .when(col("Inspection_Score") >= 0, "0-69 (Poor)")
        .otherwise("No Score")
    )
    .groupBy("score_bin")
    .count()
    .orderBy("score_bin")
)

=== Dallas: Score Distribution (binned) ===


score_bin,count
0-69 (Poor),507
70-79 (Fair),2664
80-89 (Good),25564
90-100 (Excellent),50243
No Score,6


In [0]:
# Zip code distribution (top 20)
print("=== Dallas: Top 20 Zip Codes ===")
display(
    df_dallas.groupBy("Zip_Code")
    .count()
    .orderBy(col("count").desc())
    .limit(20)
)

=== Dallas: Top 20 Zip Codes ===


Zip_Code,count
75201,4687
75220,3784
75206,3373
75243,3146
75211,3075
75217,3066
75231,2979
75229,2613
75228,2591
75208,2573


### 3.4 Dallas Violations Analysis
Note: Dallas has 25 violation groups stored as wide columns (Violation_Description_1 through 25).

In [0]:
# Count how many violations each inspection has
from pyspark.sql.functions import lit

violation_desc_cols = [c for c in df_dallas.columns if c.startswith("Violation_Description")]

df_dallas_violation_count = df_dallas.withColumn(
    "violation_count",
    sum([when(col(c).isNotNull() & (col(c) != ""), lit(1)).otherwise(lit(0)) for c in violation_desc_cols])
)

print("=== Dallas: Violations Per Inspection Distribution ===")
display(
    df_dallas_violation_count.groupBy("violation_count")
    .count()
    .orderBy("violation_count")
)

=== Dallas: Violations Per Inspection Distribution ===


violation_count,count
0,6579
1,7550
2,8941
3,8884
4,8224
5,7415
6,6562
7,5667
8,4827
9,4068


In [0]:
# Inspections with zero violations
no_violations_dallas = df_dallas_violation_count.filter(col("violation_count") == 0).count()
print(f"Dallas inspections with NO violations: {no_violations_dallas} ({no_violations_dallas/dallas_total*100:.2f}%)")

Dallas inspections with NO violations: 6579 (8.33%)


### 3.5 Dallas - Validate score >= 90 with > 3 violations (rule check)

In [0]:
high_score_many_violations = (
    df_dallas_violation_count
    .filter((col("Inspection_Score") >= 90) & (col("violation_count") > 3))
    .count()
)
print(f"Dallas records with score >= 90 AND > 3 violations: {high_score_many_violations}")
print("These rows will be flagged/dropped in Silver layer.")

Dallas records with score >= 90 AND > 3 violations: 18301
These rows will be flagged/dropped in Silver layer.


## 4. Visual Data Summarization (Built-in Profiling)
These generate rich visual profiling reports directly in the notebook.

### 4.1 Chicago - Visual Summary

In [0]:
# Read directly from CSV to avoid catalog resolution issues with summarize()
df_chicago_raw = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{VOLUME_PATH}/{CHICAGO_FILE}")
dbutils.data.summarize(df_chicago_raw)

<!DOCTYPE html>

### 4.2 Dallas - Visual Summary

In [0]:
# Read directly from CSV to avoid catalog resolution issues with summarize()
df_dallas_raw = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{VOLUME_PATH}/{DALLAS_FILE}")
dbutils.data.summarize(df_dallas_raw)

<!DOCTYPE html>

## 5. DQX Data Quality Checks
Using Databricks Labs DQX to define and apply data quality rules as expectations.

In [0]:
from databricks.labs.dqx.engine import DQEngine
from databricks.sdk import WorkspaceClient

dq_engine = DQEngine(WorkspaceClient())

### 5.1 DQX Quality Checks - Chicago

In [0]:
# Define quality rules as dicts (DQX engine format)
chicago_checks = [
    {"name": "chicago_inspection_id_not_null", "criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "Inspection_ID"}}},
    {"name": "chicago_dba_name_not_null", "criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "DBA_Name"}}},
    {"name": "chicago_inspection_date_not_null", "criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "Inspection_Date"}}},
    {"name": "chicago_results_not_null", "criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "Results"}}},
    {"name": "chicago_inspection_type_not_null", "criticality": "warn", "check": {"function": "is_not_null", "arguments": {"column": "Inspection_Type"}}},
    {"name": "chicago_zip_not_null", "criticality": "warn", "check": {"function": "is_not_null", "arguments": {"column": "Zip"}}},
    {"name": "chicago_city_not_null", "criticality": "warn", "check": {"function": "is_not_null", "arguments": {"column": "City"}}},
    {"name": "chicago_license_not_null", "criticality": "warn", "check": {"function": "is_not_null", "arguments": {"column": "License"}}},
]

chicago_valid, chicago_quarantine = dq_engine.apply_checks_by_metadata_and_split(df_chicago, chicago_checks)

print(f"Chicago - Valid rows: {chicago_valid.count()}, Quarantined rows: {chicago_quarantine.count()}")

Chicago - Valid rows: 308431, Quarantined rows: 212


In [0]:
print("=== Chicago Quarantined Rows Sample ===")
display(chicago_quarantine.limit(20))

=== Chicago Quarantined Rows Sample ===


Inspection_ID,DBA_Name,AKA_Name,License,Facility_Type,Risk,Address,City,State,Zip,Inspection_Date,Inspection_Type,Results,Violations,Latitude,Longitude,Location,data_source_name,ingest_timestamp,etl_job_id,_errors,_warnings
2633622,BON APPETIT AT OBAMA PRESIDENTIAL CENTER,OBAMA PRESIDENTIAL CENTER,3069636,Restaurant,Risk 3 (Low),6011 S STONY ISLAND AVE,null,IL,60637,2026-04-07,License,Pass w/ Conditions,null,41.78574881275,-87.58643828073,"(41.78574881274826, -87.58643828072859)",Chicago_OpenData_FoodInspections,2026-04-11T22:15:33.291Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/01_bronze_ingestion,null,"List(List(chicago_city_not_null, Column 'City' value is null, List(City), null, is_not_null, 2026-04-12T02:32:59.046Z, 75093edb-5069-4a3f-8b1b-72056c91f594, Map()))"
2633616,BON APPETIT AT OBAMA PRESIDENTIAL CENTER,OBAMA PRESIDENTIAL CENTER,3069634,Restaurant,Risk 3 (Low),6011 S STONY ISLAND AVE,null,IL,60637,2026-04-07,License,Pass w/ Conditions,null,41.78574881275,-87.58643828073,"(41.78574881274826, -87.58643828072859)",Chicago_OpenData_FoodInspections,2026-04-11T22:15:33.291Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/01_bronze_ingestion,null,"List(List(chicago_city_not_null, Column 'City' value is null, List(City), null, is_not_null, 2026-04-12T02:32:59.046Z, 75093edb-5069-4a3f-8b1b-72056c91f594, Map()))"
2633595,BON APPETIT AT OBAMA PRESIDENTIAL CENTER,OBAMA PRESIDENTIAL CENTER,3069627,Restaurant,Risk 1 (High),6011 S STONY ISLAND AVE,null,IL,60637,2026-04-07,License,Pass,58. ALLERGEN TRAINING AS REQUIRED - Comments: 2-102.13 OBSERVED NO PROOF OF ALLERGEN TRAINING FOR THE FOOD SERVICE SANITATION MANAGERS. INSTRUCTED TO PROVIDE.,41.78574881275,-87.58643828073,"(41.78574881274826, -87.58643828072859)",Chicago_OpenData_FoodInspections,2026-04-11T22:15:33.291Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/01_bronze_ingestion,null,"List(List(chicago_city_not_null, Column 'City' value is null, List(City), null, is_not_null, 2026-04-12T02:32:59.046Z, 75093edb-5069-4a3f-8b1b-72056c91f594, Map()))"
2633593,BON APPETIT AT OBAMA PRESIDENTIAL CENTER,OBAMA PRESIDENTIAL CENTER,3069628,Restaurant,Risk 3 (Low),6011 S STONY ISLAND AVE,null,IL,60637,2026-04-07,License,Pass,null,41.78574881275,-87.58643828073,"(41.78574881274826, -87.58643828072859)",Chicago_OpenData_FoodInspections,2026-04-11T22:15:33.291Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/01_bronze_ingestion,null,"List(List(chicago_city_not_null, Column 'City' value is null, List(City), null, is_not_null, 2026-04-12T02:32:59.046Z, 75093edb-5069-4a3f-8b1b-72056c91f594, Map()))"
2633617,BON APPETIT AT OBAMA PRESIDENTIAL CENTER,OBAMA PRESIDENTIAL CENTER,3069633,Restaurant,Risk 1 (High),6011 S STONY ISLAND AVE,null,IL,60637,2026-04-07,License,Pass w/ Conditions,"47. FOOD & NON-FOOD CONTACT SURFACES CLEANABLE, PROPERLY DESIGNED, CONSTRUCTED & USED - Comments: 4-201.11 OBSERVED A CRACK IN THE GLASS FOOD DISPLAY AT THE FRONT CUSTOMER COUNTER. INSTRUCTED TO REPAIR OR REPLACE IN ORDER TO AVOID ANY POSSIBLE CONTAMINATION. | 48. WAREWASHING FACILITIES: INSTALLED, MAINTAINED & USED; TEST STRIPS - Comments: 4-501.19 OBSERVED INADEQUATE HOT WATER AT THE LEFT FAUCET OF THE 3 COMPARTMENT SINK. AT THE TIME OF THE INSPECTION HOT WATER TEMPERATURE RANGE WAS 98F-100F. INSTRUCTED TO MAINTAIN 110F AT ALL TIMES. ENGINEER ADJUSTED THE VALVES AND BY THE END OF THE INSPECTION HOT WATER WAS MAINTAINING 110F. PRIORITY VIOLATION 7-38-025. | 58. ALLERGEN TRAINING AS REQUIRED - Comments: 2-102.13 OBSERVED NO ALLERGEN TRAINING CERTIFICATE FOR THE FOOD SERVICE SANITATION MANAGER. INSTRUCTED TO MEET REQUIREMENT.",41.78574881275,-87.58643828073,"(41.78574881274826, -87.58643828072859)",Chicago_OpenData_FoodInspections,2026-04-11T22:15:33.291Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/01_bronze_ingestion,null,"List(List(chicago_city_not_null, Column 'City' value is null

### 5.2 DQX Quality Checks - Dallas

In [0]:
# Define quality rules for Dallas
dallas_checks = [
    {"name": "dallas_restaurant_name_not_null", "criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "Restaurant_Name"}}},
    {"name": "dallas_inspection_date_not_null", "criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "Inspection_Date"}}},
    {"name": "dallas_inspection_type_not_null", "criticality": "error", "check": {"function": "is_not_null", "arguments": {"column": "Inspection_Type"}}},
    {"name": "dallas_zip_code_not_null", "criticality": "warn", "check": {"function": "is_not_null", "arguments": {"column": "Zip_Code"}}},
    {"name": "dallas_inspection_score_not_null", "criticality": "warn", "check": {"function": "is_not_null", "arguments": {"column": "Inspection_Score"}}},
    {"name": "dallas_street_address_not_null", "criticality": "warn", "check": {"function": "is_not_null", "arguments": {"column": "Street_Address"}}},
]

dallas_valid, dallas_quarantine = dq_engine.apply_checks_by_metadata_and_split(df_dallas, dallas_checks)

print(f"Dallas - Valid rows: {dallas_valid.count()}, Quarantined rows: {dallas_quarantine.count()}")

Dallas - Valid rows: 78973, Quarantined rows: 11


In [0]:
print("=== Dallas Quarantined Rows Sample ===")
display(dallas_quarantine.limit(20))

=== Dallas Quarantined Rows Sample ===


Restaurant_Name,Inspection_Type,Inspection_Date,Inspection_Score,Street_Number,Street_Name,Street_Direction,Street_Type,Street_Unit,Street_Address,Zip_Code,Violation_Description_1,Violation_Points_1,Violation_Detail_1,Violation_Memo_1,Violation_Description_2,Violation_Points_2,Violation_Detail_2,Violation_Memo_2,Violation_Description_3,Violation_Points_3,Violation_Detail_3,Violation_Memo_3,Violation_Description_4,Violation_Points_4,Violation_Detail_4,Violation_Memo_4,Violation_Description_5,Violation_Points_5,Violation_Detail_5,Violation_Memo_5,Violation_Description_6,Violation_Points_6,Violation_Detail_6,Violation_Memo_6,Violation_Description_7,Violation_Points_7,Violation_Detail_7,Violation_Memo_7,Violation_Description_8,Violation_Points_8,Violation_Detail_8,Violation_Memo_8,Violation_Description_9,Violation_Points_9,Violation_Detail_9,Violation_Memo_9,Violation_Description_10,Violation_Points_10,Violation_Detail_10,Violation_Memo_10,Violation_Description_11,Violation_Points_11,Violation_Detail_11,Violation_Memo_11,Violation_Description_12,Violation_Points_12,Violation_Detail_12,Violation_Memo_12,Violation_Description_13,Violation_Points_13,Violation_Detail_13,Violation_Memo_13,Violation_Description_14,Violation_Points_14,Violation_Detail_14,Violation_Memo_14,Violation_Description_15,Violation_Points_15,Violation_Detail_15,Violation_Memo_15,Violation_Description_16,Violation_Points_16,Violation_Detail_16,Violation_Memo_16,Violation_Description_17,Violation_Points_17,Violation_Detail_17,Violation_Memo_17,Violation_Description_18,Violation_Points_18,Violation_Detail_18,Violation_Memo_18,Violation_Description_19,Violation_Points_19,Violation_Detail_19,Violation_Memo_19,Violation_Description_20,Violation_Points_20,Violation_Detail_20,Violation_Memo_20,Violation_Description_21,Violation_Points_21,Violation_Detail_21,Violation_Memo_21,Violation_Description_22,Violation_Points_22,Violation_Detail_22,Violation_Memo_22,Violation_Description_23,Violation_Points_23,Violation_Detail_23,Violation_Memo_23,Violation_Description_24,Violation_Points_24,Violation_Detail_24,Violation_Memo_24,Violation_Description_25,Violation_Points_25,Violation_Detail_25,Violation_Memo_25,Inspection_Month,Inspection_Year,Lat_Long_Location,data_source_name,ingest_timestamp,etl_job_id,_errors,_warnings
null,Routine,2016-12-20,88,6449,GREENVILLE,null,AVE,null,6449 GREENVILLE AVE,75206,*45 Unnecessary articles prohibited,1,"228.186 Physical Facilities. Premises, buildings, systems, rooms, fixtures, equipment, devices, and materials. (n) Maintaining premises, unnecessary items and litter. The premises shall be free of: (1) items that are unnecessary to the operation or maintenance of the establishment such as equipment that is nonfunctional or no longer used; and",remove all unnecessary articles from backroom,"*45Floor, wall, ceiling - Exposed material",1,"(c) Floors, walls, and ceilings. (1) A food establishment containing a food handling area, food processing area, food preparation area, food storage area, equipment or utensil washing area, walk-in refrigerating unit, dressing room, locker room, toilet room, or vestibule shall: (C) prevent exposed construction in these areas, including but not limited to the exposure of pipes, conduits, ductwork, studs, joists, and rafters;",replace ceiling tile over ice machine,*21 RFSM - Not On Site,2,"Sec. 17-2.2(c)(1)(D) (c) Registered food service managers. (1) Registered food service managers required. (D) A food establishment shall have one registered food service manager employed and present in the establishment during all hours of operation, except that a registered food service manager serving multiple food establishments as authorized by Section 17-2.2(c)(1)(C) must only be present in the building in which the food establishment is located during all hours of operation.","none on site, none registered with city citation at next offense, fifth consecutive violation",*02 Cold Hold (41øF/45øF or below),3,"228.75 Food. Ti

## 6. Common Attributes Mapping (for merging)

| Concept | Chicago Column | Dallas Column | Notes |
|---|---|---|---|
| Restaurant Name | `DBA_Name` | `Restaurant_Name` | Direct mapping |
| Also Known As | `AKA_Name` | N/A | Chicago only |
| License # | `License` | N/A | Chicago only |
| Facility Type | `Facility_Type` | N/A | Chicago only |
| Risk Category | `Risk` | N/A | Chicago only |
| Inspection Date | `Inspection_Date` | `Inspection_Date` | Direct mapping |
| Inspection Type | `Inspection_Type` | `Inspection_Type` | Values may differ |
| Inspection Result | `Results` | Derived from Score | Chicago has text, Dallas needs derivation |
| Inspection Score | Derived from Results | `Inspection_Score` | Dallas has numeric, Chicago needs derivation |
| Address | `Address` | `Street_Address` | Direct mapping |
| City | `City` | Hardcode "DALLAS" | Dallas dataset doesn't have city column |
| State | `State` | Hardcode "TX" | Dallas dataset doesn't have state column |
| Zip | `Zip` | `Zip_Code` | Direct mapping |
| Latitude | `Latitude` | Parse from `Lat_Long_Location` | Dallas combines lat/long |
| Longitude | `Longitude` | Parse from `Lat_Long_Location` | Dallas combines lat/long |
| Violations | Single pipe-delimited string | 25 wide column groups | Need to standardize into rows |

## 7. Data Quality Summary

### Chicago Issues Found:
- Violations stored as unstructured pipe-delimited text (needs parsing)
- No numeric inspection score (needs derivation from Results)
- Potential null values in key fields (to be validated in Silver)

### Dallas Issues Found:
- 114 columns with wide violation format (needs unpivoting to rows)
- Missing city and state columns (hardcode Dallas, TX)
- Lat/Long combined in single field (needs parsing)
- Scores potentially > 100 (invalid, to be dropped in Silver)
- High-score inspections with too many violations (to be dropped in Silver)
- No explicit inspection result text (needs derivation from score)